In [0]:
from pyspark.sql.functions import *
from delta import DeltaTable

### Create Flag Parameter

In [0]:
dbutils.widgets.text("incemental_flag", '0')
incemental_flag = dbutils.widgets.get("incemental_flag")
print(incemental_flag)

In [0]:
silver_df = spark.read.format("parquet")\
    .load("/Volumes/adbkomal/silver/silver_data/CarSalesData")

display(silver_df)

CREATE MODEL DIMENSIONS

In [0]:
src_df = silver_df.dropDuplicates(["Dealer_ID"]).select("Dealer_ID", "DealerName")
display(src_df)

### Branch dim sink intial and incremental

In [0]:
if spark.catalog.tableExists("adbkomal.gold.dim_dealer"):
    sink_df = spark.sql("""
                        SELECT dim_dealer_key, Dealer_ID, DealerName
                        FROM adbkomal.gold.dim_dealer
                        """)
    sink_df.display()

else:
    sink_df = spark.read.format("parquet")\
            .load("/Volumes/adbkomal/silver/silver_data/CarSalesData")
    sink_df = src_df.withColumn("dim_dealer_key", lit(1))
    sink_df = sink_df.select("dim_dealer_key","Dealer_ID","DealerName").where("1 = 0")
    sink_df.display()

### Filtering new records vs old records

In [0]:
# filter_df = src_df.join(sink_df, src_df["Dealer_ID"] == sink_df["Dealer_ID"], "left").select( src_df["Dealer_ID"], src_df["DealerName"], sink_df["dim_dealer_key"])
# filter_df.display()
filter_df = src_df.alias("src").join(
    sink_df.alias("sink"), 
    col("src.Dealer_ID") == col("sink.Dealer_ID"), 
    "left"
).select(
    col("src.Dealer_ID"), 
    col("src.DealerName"), 
    col("sink.dim_dealer_key")
)


**filter old records**

In [0]:
df_filter_old = filter_df.filter(col("dim_dealer_key").isNotNull())
df_filter_new = filter_df.filter(col("dim_dealer_key").isNull()).select("Dealer_ID","DealerName")
df_filter_old.display()
df_filter_new.display()

### Fetch the max surrogate key from dim table

In [0]:
if incemental_flag == '0':
    max_value = 1
else:
    max_value = spark.sql("""
                          SELECT MAX(dim_dealer_key) as max_value
                          FROM adbkomal.gold.dim_dealer
                          """).collect()[0]['max_value'] + 1

### Create SURROGATE KEY


In [0]:
df_filter_new = df_filter_new.withColumn("dim_dealer_key", max_value + monotonically_increasing_id())
df_filter_new.display()


Create final_df = df_filter_old + df_filter_new

In [0]:
final_df = df_filter_old.union(df_filter_new)
final_df.display()
# final_df.write.mode("overwrite").saveAsTable("adbkomal.gold.dim_dealer")

### Slowly Changing Dimension - Type 1 (Upsert)

In [0]:
if spark.catalog.tableExists("adbkomal.gold.dim_dealer"):
    print("incremental run")
    delta_table = DeltaTable.forPath(spark, "abfss://gold@adfstgkomal.dfs.core.windows.net/dim_dealer")
    delta_table.alias("target")\
        .merge(final_df.alias("src"), "target.dim_dealer_key = src.dim_dealer_key")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()
    print("merge completed")
else:
    print("Initial run")
    final_df.write.format("delta")\
        .mode("overwrite")\
        .option("path","abfss://gold@adfstgkomal.dfs.core.windows.net/dim_dealer")\
        .saveAsTable("adbkomal.gold.dim_dealer")
    


In [0]:
%sql
Select * from adbkomal.gold.dim_dealer;